# 🧹 Module 3 — Class 1: Find the Dirt and Clean It (Olist Edition)

**Lesson:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/1](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1)

---

## 📖 Today's story

It's Monday morning. Your boss drops a USB stick on your desk:

> *"Here's the data dump from the e-commerce platform. 100,000 orders. Predict which ones will be late."*

You plug it in. Open the file. And... 😱 it's a mess.

- Some delivery dates are **missing**.
- Date columns are stored as **text**, not real dates.
- Some rows are **duplicates**.
- Status column has weird values like `unavailable`.

**You can't train a model on this.** Today we fix every kind of dirt before any modelling.

---

## 💡 The 80% rule

ML engineers spend **80% of their time** cleaning data. The remaining 20% is the modelling people brag about online.

**Today is the 80%.** It's the part that decides whether your model actually works.

---

## 🎯 Today's plan

1. **Setup** — download Olist, load orders + customers
2. **Inspect** — 5 commands to see what's in the data
3. **Fix dtypes** — turn text dates into real dates
4. **Investigate missing values** — BEFORE deciding what to do
5. **Check duplicates**
6. **Filter to modellable rows**
7. **Save** `orders_step1.parquet` for Class 2

## 🤖 How to run

Click a code cell → press **Shift + Enter**. Read every print output. Don't skip the markdown!

---

# 🧰 Setup — get the Olist data

Olist is a **Brazilian e-commerce dataset** on Kaggle. 100k+ real orders from 2016-2018.

### How to download (do this once)

1. Go to https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
2. Click **Download** (you may need to sign in to Kaggle — free)
3. Unzip into a folder named `olist_data/` next to this notebook

### Run this cell to verify all 9 CSVs are present:

In [ ]:
import os

expected = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'product_category_name_translation.csv',
]
for f in expected:
    print(('✅' if os.path.exists(f'olist_data/{f}') else '❌'), f)

---

## 📖 Quick reference card

Words you'll see today:

| Word | Meaning | Example |
| --- | --- | --- |
| **dtype** | Data type of a column | `int64`, `object` (= text), `datetime64` |
| **NaN** | "Not a Number" — Pandas' way of saying *missing* | empty cells |
| **NaT** | "Not a Time" — missing datetime | invalid dates |
| **coerce** | When parsing fails, put NaN/NaT instead of crashing | `pd.to_datetime(..., errors='coerce')` |
| **dropna** | Remove rows that have NaN in given columns | `df.dropna(subset=[...])` |
| **duplicated** | Find rows that are identical to other rows | `df.duplicated().sum()` |
| **value_counts** | Count how many times each value appears in a column | `df['order_status'].value_counts()` |

📌 Don't memorize. Scroll back here when stuck.

---

## Step 1 — Load the two starter tables

In [ ]:
import pandas as pd
import numpy as np

orders = pd.read_csv('olist_data/olist_orders_dataset.csv')
customers = pd.read_csv('olist_data/olist_customers_dataset.csv')

print('orders   :', orders.shape)        # expect (99441, 8)
print('customers:', customers.shape)     # expect (99441, 5)

orders.head()

## Step 2 — The 5 inspection commands

Every time you load a new dataset, run these 5 things first. **Always.**

1. `.head()` — show first 5 rows
2. `.shape` — how many rows × columns
3. `.dtypes` — what type is each column?
4. `.isna().sum()` — how many missing values per column?
5. `.describe()` — summary statistics

Don't skip. Each one tells you something different.

In [ ]:
orders.head()

In [ ]:
orders.shape

In [ ]:
orders.dtypes

In [ ]:
orders.isna().sum()

In [ ]:
orders.describe(include='all')

### 📝 Write down what you noticed (markdown cell — fill in)

After running the 5 commands above, fill in these blanks in your own words:

- **Number of rows:** _____
- **Number of columns:** _____
- **Date columns are stored as:** _____ (should be `datetime64`)
- **Column with the most NaN:** _____
- **Number of unique `order_status` values:** _____

---

## Step 3 — Fix the dtypes (parse the dates)

Right now date columns are **text**. We can't compute `delivery_days` until we convert them.

### 🍞 The recipe

```python
df['col'] = pd.to_datetime(df['col'], errors='coerce')
```

**Always use `errors='coerce'`!** It turns unparseable dates into `NaT` (Not a Time) instead of crashing the whole program. Real-world dates have typos. Defensive coding wins.

### 🍎 Kid analogy

Imagine you're sorting birthday cards from kids. Most are written like `12/05/2026`. But one kid writes `tomorrow`. 

- ❌ Without `coerce`: program crashes on the word `tomorrow`. You sort 0 cards.
- ✅ With `coerce`: that one card goes in the "unreadable" pile. You sort the other 999.

Always pick option 2 in the real world.

In [ ]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]

for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

print('✅ Dates parsed!\n')
print('New dtypes:')
print(orders[date_cols].dtypes)

---

## Step 4 — Missing values: investigate BEFORE fixing

### 🚨 The most common beginner mistake

When students see NaN, their first instinct is *"fill it with the mean"* or *"drop the row"*.

**Both are wrong by default.** First you must understand **why** the value is missing.

### Why values go missing — three categories

| Type | Example | Action |
| --- | --- | --- |
| **MCAR** (Missing Completely At Random) | A typo, a dropped sensor reading | Drop or impute — both are fine |
| **MAR** (Missing At Random, given other features) | Older customers don't fill an optional survey field | Impute conditionally |
| **MNAR** (Missing **Not** At Random — the value's existence carries info) | Order delivery date is missing because the order was **never delivered** | **Don't impute** — the missingness IS the signal |

Olist's missing delivery dates are MNAR. Let's prove it.

In [ ]:
# How many rows have missing delivery date?
missing_pct = orders['order_delivered_customer_date'].isna().mean() * 100
print(f'Missing delivery date: {missing_pct:.1f}%\n')

# What kind of orders are these?
missing_delivery = orders[orders['order_delivered_customer_date'].isna()]
print('Status of orders with no delivery date:')
print(missing_delivery['order_status'].value_counts())

### 💡 What you'll see

Roughly:
- `canceled` — order was cancelled, of course no delivery
- `unavailable` — out of stock, no delivery
- `shipped` — still in transit, not delivered yet
- `delivered` (a few) — likely data-quality issue

**These rows are NOT random.** The missingness tells us something real about the order's state.

### 📝 Decision (write yours below)

Rows with missing `order_delivered_customer_date` are mostly `___`, `___`, and `___`. They represent orders that **never reached the customer**.

Decision: **keep them in a separate file** for Module 5 segmentation analysis, but **drop them from the modelling set** because we cannot compute `is_late` without a delivery date.

---

## Step 5 — Check duplicates (the habit)

Olist is well-curated, so duplicates are rare. But the **habit** of checking matters more than the actual count today.

In [ ]:
print('Full-row duplicates:', orders.duplicated().sum())
print('Unique order_ids equals total rows:',
      orders['order_id'].nunique() == len(orders))

---

## Step 6 — Filter to modellable rows

For Module 4 we want to predict `is_late`. To compute that, we need:
- A real delivery date (`order_delivered_customer_date` not NaT)
- A real estimate (`order_estimated_delivery_date` not NaT)
- The order to actually have shipped (`order_status` ∈ {delivered, invoiced, shipped, approved})

We split into two files:
- `orders_step1.parquet` → modellable orders for M4
- `orders_undelivered.parquet` → the rest, for M5 segmentation later

In [ ]:
model_statuses = {'delivered', 'invoiced', 'shipped', 'approved'}
orders_model = orders[orders['order_status'].isin(model_statuses)].copy()

before = len(orders_model)
orders_model = orders_model.dropna(subset=['order_delivered_customer_date',
                                            'order_estimated_delivery_date'])
after = len(orders_model)

orders_undelivered = orders[~orders.index.isin(orders_model.index)].copy()

print(f'✅ Kept {after:,} rows for modelling (dropped {before - after:,} with no delivery date)')
print(f'📦 Side file: {len(orders_undelivered):,} unmodellable rows for M5 later')

---

## Step 7 — Save the cleaned starter file 💾

Class 2 will load `orders_step1.parquet` and add scaling. Class 3 adds engineered features. Class 6 produces the final `olist_clean.parquet`.

**Don't lose this file.** Every M3-M8 class builds on it.

In [ ]:
orders_model.to_parquet('orders_step1.parquet', index=False)
orders_undelivered.to_parquet('orders_undelivered.parquet', index=False)

print('✅ Saved 2 files:')
print(f'   orders_step1.parquet     → {orders_model.shape}')
print(f'   orders_undelivered.parquet → {orders_undelivered.shape}')

---

## ✅ Quick Check

1. What does `errors='coerce'` do that's different from default?
2. Why is missing `order_delivered_customer_date` *signal*, not just *noise*?
3. After today's filter you should have roughly how many rows in `orders_step1.parquet`?

## 🗺️ Where this lands in the next 6 weeks

| Module | What loads `orders_step1.parquet` (or its descendants)? |
| --- | --- |
| **M3-2** (next class) | Adds scaling, saves `orders_step2.parquet` |
| **M3-3** | Adds Haversine `distance_km` and engineered date features |
| **M3-6** (lab) | Produces the final `olist_clean.parquet` |
| **M4** | Trains classifier on `is_late` — loads the M3-6 output |
| **M5** | Customer segmentation — loads the M3-6 output |
| **M6** | Neural network — loads the M3-6 output |
| **M7** | Joins review sentiment back to the M3-6 output |
| **M8** | Deploys the model — same file, same schema |

**One dataset. Six modules. One CV bullet.**

## 📤 Submit

### Bronze (Type 2 — beginner)
1. Run every cell. No errors.
2. Fill in the markdown cells where I asked for your observations.
3. Push your notebook + `orders_step1.parquet` to `module-3/class_1/submissions/<YourName>/`.

### Gold (Type 1 — comfortable, wants stretch)
1. Also clean the `customers` table (lowercase state codes, drop duplicate `customer_unique_id` rows keeping the most recent).
2. Merge `orders` + `customers` on `customer_id`. Save as `orders_customers_step1.parquet`.
3. Half-page markdown rationale: every cleaning decision you made and why.

🧹 *Great work — the data is no longer a mess. See you in Class 2 for scaling!*